# Calibration Analysis of Binary Event Contracts in Prediction Markets

**Author:** Jay Guwalani | M.S. Data Science, University of Maryland, College Park

**Date:** March 2026

This notebook provides an end-to-end analysis of whether prediction market contract prices accurately reflect outcome probabilities, with focus on the **favorite-longshot bias** documented in CFTC-regulated exchanges.

---

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.figsize': (8, 5),
    'figure.dpi': 120
})

ACCENT = '#2B5C8A'
RED = '#C0392B'
GREEN = '#27AE60'
GRAY = '#7F8C8D'
DARK = '#2C3E50'

print('Setup complete.')

## 2. Data Generation

We generate synthetic binary event contracts calibrated to empirically documented patterns from CFTC-regulated prediction markets (Burgi, Deng & Whelan, 2026). The dataset models 19 price buckets with win rates reflecting the **favorite-longshot bias**: low-price contracts win less than their price implies, high-price contracts win more.

In [ ]:
np.random.seed(42)

# Price buckets and empirically calibrated win rates
price_buckets = list(range(5, 100, 5))

actual_win_rates = {
    5: 0.021, 10: 0.058, 15: 0.098, 20: 0.145, 25: 0.198,
    30: 0.255, 35: 0.310, 40: 0.370, 45: 0.430, 50: 0.500,
    55: 0.572, 60: 0.640, 65: 0.705, 70: 0.762, 75: 0.815,
    80: 0.862, 85: 0.908, 90: 0.952, 95: 0.978
}

contracts_per_bucket = {
    5: 12400, 10: 8900, 15: 7200, 20: 6100, 25: 5400,
    30: 4800, 35: 4500, 40: 4200, 45: 4000, 50: 3800,
    55: 4000, 60: 4200, 65: 4500, 70: 4800, 75: 5400,
    80: 6100, 85: 7200, 90: 8900, 95: 12400
}

categories = ['economics', 'politics', 'weather', 'sports', 'culture']

records = []
for price in price_buckets:
    n = contracts_per_bucket[price]
    win_rate = actual_win_rates[price]
    outcomes = np.random.binomial(1, win_rate, n)
    for outcome in outcomes:
        records.append({
            'price_cents': price,
            'implied_prob': price / 100.0,
            'outcome': int(outcome),
            'category': np.random.choice(categories, p=[0.30, 0.25, 0.20, 0.15, 0.10]),
            'volume_usd': round(np.random.lognormal(mean=3.5, sigma=1.2), 2)
        })

df = pd.DataFrame(records)
print(f'Total contracts: {len(df):,}')
print(f'Categories: {df["category"].value_counts().to_dict()}')
df.head(10)

## 3. Calibration Statistics by Price Bucket

In [ ]:
summary = df.groupby('price_cents').agg(
    n_contracts=('outcome', 'count'),
    wins=('outcome', 'sum'),
    total_volume=('volume_usd', 'sum')
).reset_index()

summary['implied_prob'] = summary['price_cents'] / 100.0
summary['actual_win_rate'] = summary['wins'] / summary['n_contracts']
summary['calibration_error'] = summary['actual_win_rate'] - summary['implied_prob']
summary['calibration_error_pp'] = summary['calibration_error'] * 100
summary['expected_roi_pct'] = (summary['actual_win_rate'] / summary['implied_prob'] - 1) * 100

# Binomial 95% CI
summary['ci_lower'] = summary.apply(
    lambda r: stats.binom.ppf(0.025, int(r['n_contracts']), r['actual_win_rate']) / r['n_contracts'], axis=1
)
summary['ci_upper'] = summary.apply(
    lambda r: stats.binom.ppf(0.975, int(r['n_contracts']), r['actual_win_rate']) / r['n_contracts'], axis=1
)

display_cols = ['price_cents', 'n_contracts', 'implied_prob', 'actual_win_rate', 
                'calibration_error_pp', 'expected_roi_pct']
summary[display_cols].round(2)

## 4. Figure 1: Calibration Plot

A perfectly calibrated market produces data points along the 45-degree line. Deviation below = overpricing (longshots), above = underpricing (favorites).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = summary['implied_prob'].values
y = summary['actual_win_rate'].values

ax.plot([0, 1], [0, 1], '--', color=GRAY, linewidth=1, label='Perfect Calibration')
ax.scatter(x, y, c=ACCENT, s=60, zorder=3, edgecolors='white', linewidth=0.5)
ax.plot(x, y, '-', color=ACCENT, linewidth=1.5, alpha=0.7, label='Observed Win Rate')

midpoint = len(x) // 2
ax.fill_between(x[:midpoint+1], x[:midpoint+1], y[:midpoint+1],
                alpha=0.15, color=RED, label='Longshot Overpricing')
ax.fill_between(x[midpoint:], x[midpoint:], y[midpoint:],
                alpha=0.15, color=GREEN, label='Favorite Underpricing')

# Add CI bands
ax.fill_between(x, summary['ci_lower'], summary['ci_upper'], alpha=0.1, color=ACCENT)

ax.set_xlabel('Contract Price (Implied Probability)')
ax.set_ylabel('Actual Win Rate')
ax.set_title('Calibration Analysis: Favorite-Longshot Bias in Prediction Markets')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Figure 2: Expected ROI by Contract Price

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

prices = summary['price_cents'].values
roi = summary['expected_roi_pct'].values
bar_colors = [RED if r < 0 else GREEN for r in roi]

ax.bar(prices, roi, width=4, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(y=0, color=DARK, linewidth=0.8)
ax.set_xlabel('Contract Price (cents)')
ax.set_ylabel('Expected ROI (%)')
ax.set_title('Expected Return on Investment by Contract Price')
ax.set_xticks(prices[::2])
plt.tight_layout()
plt.show()

## 6. Figure 3: Calibration Error Decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

errors = summary['calibration_error_pp'].values
error_colors = [RED if e < 0 else GREEN for e in errors]

ax.bar(prices, errors, width=4, color=error_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(y=0, color=DARK, linewidth=0.8)
ax.set_xlabel('Contract Price (cents)')
ax.set_ylabel('Calibration Error (percentage points)')
ax.set_title('Calibration Error by Contract Price Bucket')
ax.set_xticks(prices[::2])
plt.tight_layout()
plt.show()

## 7. Statistical Validation

In [ ]:
# Brier Score
brier = np.mean((df['implied_prob'] - df['outcome']) ** 2)
print(f'Brier Score: {brier:.4f} (lower = better calibrated)')

# Chi-squared goodness-of-fit
expected_wins = summary['implied_prob'] * summary['n_contracts']
observed_wins = summary['wins']
scale = observed_wins.sum() / expected_wins.sum()
chi2_stat, chi2_p = stats.chisquare(observed_wins, expected_wins * scale)
print(f'Chi-squared: {chi2_stat:.1f} (p = {chi2_p:.2e})')
print(f'Calibration is {"significantly biased" if chi2_p < 0.05 else "not significantly biased"}')

# Calibration regression: win_rate = a + b * implied_prob
slope, intercept, r_value, p_value, std_err = stats.linregress(
    summary['implied_prob'], summary['actual_win_rate']
)
print(f'\nCalibration Regression:')
print(f'  Win Rate = {intercept:.4f} + {slope:.4f} * Implied Prob')
print(f'  R-squared: {r_value**2:.4f}')
print(f'  Slope: {slope:.4f} (perfect calibration = 1.000)')
print(f'  Intercept: {intercept:.4f} (perfect calibration = 0.000)')

# Test if slope significantly differs from 1
t_stat = (slope - 1) / std_err
p_slope = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(summary) - 2))
print(f'  H0 (slope=1): t={t_stat:.2f}, p={p_slope:.4f}')

## 8. Bootstrap ROI Analysis

In [ ]:
np.random.seed(42)
n_bootstrap = 10000
boot_results = []

for price, group in df.groupby('price_cents'):
    outcomes = group['outcome'].values
    implied = price / 100.0
    boot_rois = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(outcomes, size=len(outcomes), replace=True)
        boot_rois.append((sample.mean() / implied - 1) * 100)
    boot_rois = np.array(boot_rois)
    boot_results.append({
        'price_cents': price,
        'roi_mean': boot_rois.mean(),
        'roi_ci_lower': np.percentile(boot_rois, 2.5),
        'roi_ci_upper': np.percentile(boot_rois, 97.5)
    })

boot_df = pd.DataFrame(boot_results)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(boot_df['price_cents'], boot_df['roi_mean'], width=4,
       color=[RED if r < 0 else GREEN for r in boot_df['roi_mean']], alpha=0.7)
ax.errorbar(boot_df['price_cents'], boot_df['roi_mean'],
            yerr=[boot_df['roi_mean'] - boot_df['roi_ci_lower'],
                  boot_df['roi_ci_upper'] - boot_df['roi_mean']],
            fmt='none', color=DARK, capsize=2, linewidth=0.8)
ax.axhline(y=0, color=DARK, linewidth=0.8)
ax.set_xlabel('Contract Price (cents)')
ax.set_ylabel('Expected ROI (%) with 95% CI')
ax.set_title('Bootstrap ROI Analysis (10,000 iterations)')
ax.set_xticks(boot_df['price_cents'][::2])
plt.tight_layout()
plt.show()

print('\nBootstrap ROI Summary:')
for _, r in boot_df.iterrows():
    print(f"  {int(r['price_cents']):3d}c: {r['roi_mean']:+6.1f}% [{r['roi_ci_lower']:+6.1f}%, {r['roi_ci_upper']:+6.1f}%]")

## 9. Category-Level Calibration

In [ ]:
cat_results = []
for cat, group in df.groupby('category'):
    cat_summary = group.groupby('price_cents').agg(
        n=('outcome', 'count'), wins=('outcome', 'sum')
    ).reset_index()
    cat_summary['implied'] = cat_summary['price_cents'] / 100.0
    cat_summary['actual'] = cat_summary['wins'] / cat_summary['n']
    brier_cat = np.mean((group['implied_prob'] - group['outcome']) ** 2)
    mace = (cat_summary['actual'] - cat_summary['implied']).abs().mean()
    cat_results.append({'category': cat, 'n_contracts': len(group),
                        'brier_score': brier_cat, 'mean_abs_cal_error': mace})

cat_df = pd.DataFrame(cat_results).sort_values('brier_score')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(cat_df['category'], cat_df['brier_score'], color=ACCENT, alpha=0.8)
axes[0].set_xlabel('Brier Score')
axes[0].set_title('Calibration Quality by Category')

axes[1].barh(cat_df['category'], cat_df['mean_abs_cal_error'] * 100, color=ACCENT, alpha=0.8)
axes[1].set_xlabel('Mean Absolute Calibration Error (pp)')
axes[1].set_title('Average Miscalibration by Category')

plt.tight_layout()
plt.show()

cat_df.round(4)

## 10. Key Findings

### Summary

| Finding | Value |
|---------|-------|
| Total contracts analyzed | 118,800 |
| Brier Score | 0.1222 |
| Chi-squared (calibration test) | p < 0.001 (significant) |
| Longshot (5-25c) avg ROI | -38.7% |
| Favorite (75-95c) avg ROI | +6.6% |
| Calibration regression slope | 1.15 (perfect = 1.00) |

### Implications for Market Design

1. **Forecasting accuracy**: Raw contract prices are informative but systematically biased at the extremes. Recalibration methods (isotonic regression, Platt scaling) could improve information value.

2. **Market microstructure**: The maker-taker structure may contribute to the bias. Analyzing whether bias differs for maker-initiated vs. taker-initiated trades would help isolate the mechanism.

3. **Fee structure impact**: Transaction fees disproportionately affect cheap contracts, potentially amplifying the longshot bias. Fee-adjusted calibration analysis would be a natural extension.

### Future Work

- Connect to Kalshi's public API for live data analysis
- Track calibration improvement as contracts approach expiry
- Compare bias across event categories with larger sample sizes
- Implement recalibration models and measure improvement in Brier score

---

**References**

- Arrow, K. J., et al. (2008). The Promise of Prediction Markets. *Science*, 320(5878), 877-878.
- Burgi, C., Deng, W., & Whelan, K. (2026). Makers and Takers: The Economics of the Kalshi Prediction Market. *CEPR Discussion Paper No. 20631*.
- Ottaviani, M., & Sorensen, P. N. (2008). The Favorite-Longshot Bias. In *Handbook of Sports and Lottery Markets*, Elsevier.